# Lesson 16 — Decision Trees and Random Forests

**What this notebook does:** builds a decision tree completely from scratch — a model that routes a support ticket to the right team (shipping, billing, or account) by asking yes/no questions about the words the ticket contains. We measure how good a question is with Gini impurity, grow the tree by recursion, watch how depth controls overfitting, and then grow a small **random forest** to see why many trees voting together beat one tree. Plain Python only — no external libraries.

## Step 1 — The training data

Same starting point as every supervised-learning lesson: **past tickets a human already labelled**. This time the label is the **category** — which team should handle the ticket. This is the task we framed back in Lesson 11, now finally built for real.

The important shift from the last two lessons: linear and logistic regression learned **numbers** (a weight and a bias). A decision tree learns something different — **which questions to ask, and in what order** — entirely from these 12 examples.

In [1]:
# 12 past tickets, each labelled with the team that handled it
TRAINING_TICKETS = [
    ("Where is my package it has not arrived", "shipping"),
    ("My order says delivered but no package came", "shipping"),
    ("The delivery is late and tracking has not updated", "shipping"),
    ("My package arrived damaged and crushed", "shipping"),
    ("I was charged twice for my order please refund me", "billing"),
    ("I want a refund for this purchase", "billing"),
    ("Why was I charged extra fees I want a refund", "billing"),
    ("Requesting a refund because the discount was not applied", "billing"),
    ("My login is not working after the update", "account"),
    ("I forgot my password and the reset email never comes", "account"),
    ("Please help me change the email on my account", "account"),
    ("My account is locked and I need to reset my password", "account"),
]

# count how many tickets each category has, to see the class balance
category_counts = {}
for text, label in TRAINING_TICKETS:
    if label not in category_counts:
        category_counts[label] = 0    # first time we see this label
    category_counts[label] += 1       # one more ticket for this label

print("Tickets per category:")
for label in category_counts:
    print("  " + label + ":", category_counts[label])

Tickets per category:
  shipping: 4
  billing: 4
  account: 4


## Step 2 — The questions the tree is allowed to ask

A decision tree can only ask **yes/no questions**. For text, the simplest useful question is: *"does the ticket contain the word W?"*

We hand the tree a small menu of candidate words. Choosing what a model gets to look at is called **feature engineering** — Chapter 6 is entirely about it. What the tree must figure out **on its own**, from the data, is *which* of these questions actually separate the categories, and *in what order* to ask them. No human ranks the words.

Note the deliberate weakness we are building in: the question is about the **exact** word. "delivery" and "delivered" are different words to this model. We will watch that bite us later in the notebook.

In [2]:
# the menu of yes/no questions: "does the ticket contain this word?"
CANDIDATE_WORDS = [
    "refund", "charged", "package", "delivery", "order",
    "password", "account", "login", "help",
]


def tokenize(text):
    # lowercase, strip basic punctuation, split on spaces
    lowered = text.lower().replace(",", "").replace(".", "").replace("!", "").replace("?", "")
    return lowered.split()


def has_word(text, word):
    # True if the ticket contains this exact word
    return word in tokenize(text)


# demo: which questions does one billing ticket answer yes to?
demo_text = "I was charged twice for my order please refund me"
print("Ticket:", demo_text)
for word in CANDIDATE_WORDS:
    answer = has_word(demo_text, word)
    print('  contains "' + word + '"?', answer)

Ticket: I was charged twice for my order please refund me
  contains "refund"? True
  contains "charged"? True
  contains "package"? False
  contains "delivery"? False
  contains "order"? True
  contains "password"? False
  contains "account"? False
  contains "login"? False
  contains "help"? False


## Step 3 — Gini impurity: a score for how mixed a pile is

To choose good questions, the tree needs a way to *measure messiness*. "This pile looks better sorted" is a feeling — a program needs a number.

**Plain definition:** take a pile of labelled tickets. Pick one ticket at random, note its label, put it back, and pick again. The chance that the two labels **disagree** is the pile's impurity.

**Worked example, step by step** — a pile of 4 tickets, 2 shipping and 2 billing:

1. **Each label's share of the pile.** A share is just: tickets with that label ÷ tickets in the pile. Shipping: 2/4 = 0.5. Billing: 2/4 = 0.5.
2. **Chance two random picks land on the SAME label.** Both picks shipping: 0.5 × 0.5 = 0.25. Both billing: 0.5 × 0.5 = 0.25. This is why each share gets **squared** — squaring is "the chance of that label twice in a row".
3. **Chance the two picks agree:** 0.25 + 0.25 = 0.5.
4. **Flip it — chance they disagree:** 1 − 0.5 = **0.5**. That is the impurity.

As one formula: 1 − (1/2)² − (1/2)² = **0.5**.

Check the extremes: a pure pile (all shipping) gives 1 − 1² = **0.0** — two picks can never disagree. Our full 12-ticket pile (a third each) gives 1 − 3 × (1/3)² = **0.667** — very mixed.

**Technical name:** this is **Gini impurity**: *1 minus the sum, over the labels, of each label's share squared*.

Rule of thumb: **0 = perfectly sorted, bigger = messier.**

In [3]:
def gini(labels):
    total = len(labels)
    # count how many times each label appears
    counts = {}
    for label in labels:
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    # start from 1 and subtract each label's squared share
    impurity = 1.0
    for label in counts:
        share = counts[label] / total
        impurity -= share * share
    return impurity


# a pure pile: everyone agrees, impurity is zero
pure_pile = ["billing", "billing", "billing", "billing"]
print("pure pile:", round(gini(pure_pile), 3))

# a 50/50 pile: half the random pairs disagree
half_pile = ["shipping", "shipping", "account", "account"]
print("50/50 pile:", round(gini(half_pile), 3))

# our full training pile: three categories, maximally mixed
all_labels = []
for text, label in TRAINING_TICKETS:
    all_labels.append(label)
print("all 12 tickets:", round(gini(all_labels), 3))

pure pile: 0.0
50/50 pile: 0.5
all 12 tickets: 0.667


## Step 4 — Scoring a question

A question splits the 12 tickets into two piles: the **YES pile** (tickets containing the word) and the **NO pile** (the rest). A good question leaves both piles *purer* than the pile we started with.

The score of a question is the **weighted average of the two piles' Gini impurities** — each pile counts in proportion to how many tickets it holds:

> score = (share of tickets in YES pile) × (gini of YES pile) + (share in NO pile) × (gini of NO pile)

**Lower is better** (less leftover messiness). The weighting matters: a question that makes a tiny pile perfectly pure but leaves a huge messy pile has barely helped, and the weighting makes sure the huge pile dominates the score.

Worked example for *contains "refund"?* on our 12 tickets:

- YES pile: 4 tickets, all billing → gini 0.0
- NO pile: 8 tickets, 4 shipping + 4 account → gini = 1 − (1/2)² − (1/2)² = 0.5
- score = (4/12) × 0.0 + (8/12) × 0.5 = **0.333**

Starting messiness was 0.667, so this one question removed half of it in a single cut.

One edge case: if a word appears in *none* (or *all*) of the pile's tickets, the "split" puts everything on one side and separates nothing. We return `None` for those so they can be skipped.

In [4]:
def split_quality(tickets, word):
    # sort every ticket's label onto the yes side or the no side
    yes_labels = []
    no_labels = []
    for text, label in tickets:
        if has_word(text, word):
            yes_labels.append(label)
        else:
            no_labels.append(label)
    # a question that puts everything on one side tells us nothing
    if len(yes_labels) == 0 or len(no_labels) == 0:
        return None
    # weight each side by its share of the tickets
    total = len(tickets)
    yes_weight = len(yes_labels) / total
    no_weight = len(no_labels) / total
    # weighted average of the two sides' impurities
    score = yes_weight * gini(yes_labels) + no_weight * gini(no_labels)
    return score


# score three very different questions and show their yes/no piles
for word in ["refund", "order", "delivery"]:
    yes_labels = []
    no_labels = []
    for text, label in TRAINING_TICKETS:
        if has_word(text, word):
            yes_labels.append(label)
        else:
            no_labels.append(label)
    print('Question: contains "' + word + '"?')
    print("  YES pile:", yes_labels)
    print("    gini:", round(gini(yes_labels), 3))
    print("  NO  pile:", no_labels)
    print("    gini:", round(gini(no_labels), 3))
    print("  weighted score:", round(split_quality(TRAINING_TICKETS, word), 3))
    print()

Question: contains "refund"?
  YES pile: ['billing', 'billing', 'billing', 'billing']
    gini: 0.0
  NO  pile: ['shipping', 'shipping', 'shipping', 'shipping', 'account', 'account', 'account', 'account']
    gini: 0.5
  weighted score: 0.333

Question: contains "order"?
  YES pile: ['shipping', 'billing']
    gini: 0.5
  NO  pile: ['shipping', 'shipping', 'shipping', 'billing', 'billing', 'billing', 'account', 'account', 'account', 'account']
    gini: 0.66
  weighted score: 0.633

Question: contains "delivery"?
  YES pile: ['shipping']
    gini: 0.0
  NO  pile: ['shipping', 'shipping', 'shipping', 'billing', 'billing', 'billing', 'billing', 'account', 'account', 'account', 'account']
    gini: 0.661
  weighted score: 0.606



## Step 5 — Let the data choose the question

Now the tree can act like the smart 20-questions player: **try every question on the menu, keep the one with the lowest score.** No human decides that "refund" is a billing word — the data does.

One detail to notice in the code: `best_word` takes the menu (`allowed_words`) as a parameter instead of always using the full `CANDIDATE_WORDS` list. For our single tree we pass the full menu. The parameter earns its keep in the random-forest section later, where each tree deliberately gets a *smaller* menu.

In [5]:
def best_word(tickets, allowed_words):
    best = None
    best_score = None
    for word in allowed_words:
        score = split_quality(tickets, word)
        if score is None:
            continue                      # question splits nothing here, skip it
        if best_score is None or score < best_score:
            best = word                   # new best question so far
            best_score = score
    return best


# score every question on the menu, at the root (all 12 tickets)
print("Scores at the root (lower = better):")
for word in CANDIDATE_WORDS:
    score = split_quality(TRAINING_TICKETS, word)
    print("  " + word + ":", round(score, 3))

print()
print("Best first question:", best_word(TRAINING_TICKETS, CANDIDATE_WORDS))

Scores at the root (lower = better):
  refund: 0.333
  charged: 0.533
  package: 0.444
  delivery: 0.606
  order: 0.633
  password: 0.533
  account: 0.533
  login: 0.606
  help: 0.606

Best first question: refund


## Step 6 — Growing the whole tree: recursion and stopping rules

One question rarely finishes the job. After splitting on "refund", the YES pile is done (all billing) — but the NO pile still mixes shipping and account tickets. So we treat that pile as a **brand-new, smaller problem** and ask again: *of all the candidate questions, which best splits THIS pile?* Then again on the piles that creates, and so on. A function that solves a problem by calling itself on smaller pieces of the problem is using **recursion** — that is exactly what `build_tree` does below.

Some vocabulary, since the tree metaphor is standard everywhere:

- A **node** is one question box in the tree.
- The **root** is the first question at the top.
- A **branch** is the yes-path or no-path leaving a question.
- A **leaf** is an end point that asks nothing and just states a prediction.
- **Depth** is how many questions deep a path goes.

When do we stop splitting and place a leaf? Three stopping rules:

1. **The pile is pure** (Gini 0) — nothing left to separate.
2. **We hit the depth limit** (`max_depth`) — our main defense against overfitting, coming up in Step 9.
3. **No question splits the pile** — every candidate word puts all tickets on one side, so asking anything is useless.

A leaf predicts the **majority label** of the tickets that landed in it. If a leaf holds 1 shipping and 4 account tickets, it predicts `account` — the best it can do without asking more questions.

One more word you will meet in every textbook: this algorithm is **greedy**. At each node it picks the question that looks best *right now*, and never reconsiders. It could miss a cleverer pair of questions that only pays off two levels later. Greedy is not optimal — but it is fast, simple, and works well in practice, and every real tree library grows trees this way.

In [6]:
def majority_label(tickets):
    # count how many tickets carry each label
    counts = {}
    for text, label in tickets:
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    # find the label with the highest count
    winner = None
    winner_count = 0
    for label in counts:
        if counts[label] > winner_count:
            winner = label
            winner_count = counts[label]
    return winner

# Compact alternative (same result):
# winner = max(counts, key=counts.get)


def build_tree(tickets, allowed_words, depth, max_depth):
    # collect the labels in this pile
    labels = []
    for text, label in tickets:
        labels.append(label)
    # stopping rule 1: the pile is already pure
    # stopping rule 2: we have reached the depth limit
    if gini(labels) == 0.0 or depth == max_depth:
        return {"label": majority_label(tickets)}
    # otherwise find the best question for THIS pile
    word = best_word(tickets, allowed_words)
    # stopping rule 3: no question splits this pile at all
    if word is None:
        return {"label": majority_label(tickets)}
    # split the pile into the yes side and the no side
    yes_tickets = []
    no_tickets = []
    for text, label in tickets:
        if has_word(text, word):
            yes_tickets.append((text, label))
        else:
            no_tickets.append((text, label))
    # recurse: grow a subtree on each side, one level deeper
    return {
        "word": word,
        "yes": build_tree(yes_tickets, allowed_words, depth + 1, max_depth),
        "no": build_tree(no_tickets, allowed_words, depth + 1, max_depth),
    }


def format_tree(node, indent=""):
    # a leaf just states its prediction
    if "label" in node:
        return "predict " + node["label"]
    # a question node shows its question, then both branches indented below
    yes_part = format_tree(node["yes"], indent + "  ")
    no_part = format_tree(node["no"], indent + "  ")
    return ('contains "' + node["word"] + '"?'
            + "\n" + indent + "  yes: " + yes_part
            + "\n" + indent + "  no:  " + no_part)


# grow the full tree, at most 3 questions deep
tree = build_tree(TRAINING_TICKETS, CANDIDATE_WORDS, 0, 3)
print("The learned tree:")
print(format_tree(tree))

The learned tree:
contains "refund"?
  yes: predict billing
  no:  contains "package"?
    yes: predict shipping
    no:  contains "delivery"?
      yes: predict shipping
      no:  predict account


## Step 7 — Reading the tree, and one suspicious branch

You should see this tree:

```text
contains "refund"?
  yes: predict billing
  no:  contains "package"?
    yes: predict shipping
    no:  contains "delivery"?
      yes: predict shipping
      no:  predict account
```

Read it top to bottom, exactly like a flowchart a human would write — except no human wrote it. The data chose every question:

- **"refund"** first, because it cleanly isolates all four billing tickets in one cut (score 0.333 — the best available).
- **"package"** next, because among the remaining 8 tickets it isolates three shipping tickets.
- **"delivery"** last, to separate the one leftover shipping ticket from the four account tickets.

Two things deserve a closer look:

1. **The `delivery` branch exists to rescue exactly ONE training ticket.** A whole branch grown for a single example should make you suspicious — it might be real signal, or it might be the tree memorizing a quirk. This is what overfitting *looks like* in a tree, and we return to it in Step 9.
2. **`account` is the "none of the above" leaf.** Any ticket that mentions none of the tree's three chosen words lands there — including tickets that have nothing to do with accounts. The tree has no idea what "account-ness" is; it only knows that after ruling out refund/package/delivery words, the leftover training tickets were mostly account ones.

To make a prediction, we just walk a ticket down the tree, answering each question, until we hit a leaf.

In [7]:
def predict(tree, ticket_text):
    node = tree
    # keep answering questions until we land on a leaf
    while "label" not in node:
        if has_word(ticket_text, node["word"]):
            node = node["yes"]    # answer was yes: follow the yes branch
        else:
            node = node["no"]     # answer was no: follow the no branch
    return node["label"]          # the leaf's label is the prediction


# check the tree against every ticket it was trained on
correct = 0
for text, label in TRAINING_TICKETS:
    prediction = predict(tree, text)
    if prediction == label:
        marker = "OK  "
        correct += 1
    else:
        marker = "MISS"
    print(" ", marker, "predicted:", prediction.ljust(8), "| actual:", label.ljust(8), "|", text)

print()
print("Training accuracy:", correct, "out of", len(TRAINING_TICKETS))

# Compact alternative (same result):
# correct = sum(1 for text, label in TRAINING_TICKETS if predict(tree, text) == label)

  OK   predicted: shipping | actual: shipping | Where is my package it has not arrived
  OK   predicted: shipping | actual: shipping | My order says delivered but no package came
  OK   predicted: shipping | actual: shipping | The delivery is late and tracking has not updated
  OK   predicted: shipping | actual: shipping | My package arrived damaged and crushed
  OK   predicted: billing  | actual: billing  | I was charged twice for my order please refund me
  OK   predicted: billing  | actual: billing  | I want a refund for this purchase
  OK   predicted: billing  | actual: billing  | Why was I charged extra fees I want a refund
  OK   predicted: billing  | actual: billing  | Requesting a refund because the discount was not applied
  OK   predicted: account  | actual: account  | My login is not working after the update
  OK   predicted: account  | actual: account  | I forgot my password and the reset email never comes
  OK   predicted: account  | actual: account  | Please help me chang

## Step 8 — New, unseen tickets — including one that fools the tree

The real test is tickets the tree never saw. The first three below are ordinary cases. The fourth is a trap we set on purpose: *"The delivery fee was charged to the wrong card"* is really a **billing** problem — but it contains the word "delivery" and does not contain the word "refund".

Trace it by hand: contains "refund"? no → contains "package"? no → contains "delivery"? **yes** → the tree says **shipping**. Wrong.

This is the honest weakness of asking about exact words: the tree keys on *which words appear*, not on *what the sentence means*. A human reads "charged to the wrong card" and instantly knows it is about money; our tree never asks about "card" because that word is not even on its menu. Keep this failure in mind — better features (Chapter 6) and, much later, models that read meaning rather than exact words (Phase 4 onward) are the answers to exactly this.

In [8]:
# four tickets the tree has never seen
new_tickets = [
    "My package never arrived and tracking is stuck",
    "I was charged for an order I cancelled please refund me",
    "I cannot reset my password on my account",
    "The delivery fee was charged to the wrong card",
]

# route each one through the tree
for text in new_tickets:
    category = predict(tree, text)
    print("  [" + category + "]", text)

  [shipping] My package never arrived and tracking is stuck
  [billing] I was charged for an order I cancelled please refund me
  [account] I cannot reset my password on my account
  [shipping] The delivery fee was charged to the wrong card


## Step 9 — Depth is the overfitting knob

Lesson 13 taught us that a model can *memorize* instead of *learn*. For trees, this shows up in a very visible way: **left unlimited, a tree will keep splitting until every leaf is pure** — which means it will happily grow a special branch for every individual quirky ticket. Perfect training accuracy, learned nothing general. That is overfitting in its purest form.

The main defense is the **depth limit**: how many questions deep the tree may go. Let's rebuild the tree with `max_depth = 2` and see what changes.

**What to expect:** the `delivery` rescue branch is no longer allowed to exist. The pile it used to fix (1 shipping ticket + 4 account tickets) becomes a leaf, and the leaf predicts its majority label: `account`. So the shallow tree gets that one delivery ticket *wrong* — training accuracy drops from 12/12 to 11/12 — but it has stopped memorizing a one-ticket quirk. On new tickets, that trade is often a win. Lower training accuracy, better model: exactly the bias-variance story from Lesson 13.

In [9]:
# same data, same menu of questions - but only 2 questions deep
shallow_tree = build_tree(TRAINING_TICKETS, CANDIDATE_WORDS, 0, 2)
print("The depth-2 tree:")
print(format_tree(shallow_tree))
print()

# score the shallow tree on the training tickets
correct = 0
for text, label in TRAINING_TICKETS:
    prediction = predict(shallow_tree, text)
    if prediction == label:
        correct += 1
    else:
        # show exactly which ticket the shallow tree gives up on
        print("  now missed:", text, "(actual: " + label + ", predicted: " + prediction + ")")

print()
print("Training accuracy:", correct, "out of", len(TRAINING_TICKETS))

The depth-2 tree:
contains "refund"?
  yes: predict billing
  no:  contains "package"?
    yes: predict shipping
    no:  predict account

  now missed: The delivery is late and tracking has not updated (actual: shipping, predicted: account)

Training accuracy: 11 out of 12


## Step 10 — One tree is fragile; a forest is stable

There is a second problem with single trees, beyond overfitting: they are **unstable**. Change a couple of training tickets and the best root question can flip — and because every later split depends on the first one, the *whole tree* can come out different. One tree is one opinion, formed by a chain of greedy choices.

The fix is simple: **train many slightly-different trees and let them vote.** This is called a **random forest**. Each tree is made different on purpose, in two ways:

1. **A bootstrap sample.** Instead of training on the 12 tickets, each tree trains on 12 tickets *drawn at random with replacement* — so some tickets appear twice and some not at all. It is like showing each tree a slightly different version of history. (Statisticians call drawing-with-replacement a *bootstrap sample*, and this train-on-resamples-then-combine trick is called **bagging**, short for bootstrap aggregating.)
2. **A restricted menu of questions.** Each tree only gets a random subset of the candidate words (here: 6 of the 9). This stops every tree from grabbing the same dominant first question and forces the forest to discover *different* routes to the answer.

Why does voting help? Each tree makes mistakes, but because the trees saw different data and different questions, they make **different** mistakes. When they vote, the errors point in different directions and largely cancel, while the true signal — which all of them partly captured — adds up. This is the same reason averaging many noisy measurements gives a better answer than trusting one measurement.

One honest simplification: a real random forest re-draws the random menu of questions **at every split**. We draw one menu per tree to keep the code short — the idea is the same.

In [10]:
import random

# seeded random generator: rerunning the notebook gives the same draws
rng = random.Random(42)


def bootstrap_sample(tickets, rng):
    sample = []
    for _ in range(len(tickets)):
        index = rng.randrange(len(tickets))   # pick one ticket index at random
        sample.append(tickets[index])         # with replacement: repeats are allowed
    return sample


# draw one bootstrap sample and look at it
sample = bootstrap_sample(TRAINING_TICKETS, rng)
print("One bootstrap sample of our 12 tickets (note the repeats and the missing ones):")
for text, label in sample:
    print("  [" + label + "]", text)

One bootstrap sample of our 12 tickets (note the repeats and the missing ones):
  [account] Please help me change the email on my account
  [shipping] My order says delivered but no package came
  [shipping] Where is my package it has not arrived
  [account] My account is locked and I need to reset my password
  [billing] I was charged twice for my order please refund me
  [shipping] My package arrived damaged and crushed
  [shipping] My package arrived damaged and crushed
  [shipping] The delivery is late and tracking has not updated
  [account] My account is locked and I need to reset my password
  [shipping] My order says delivered but no package came
  [account] Please help me change the email on my account
  [account] My account is locked and I need to reset my password


## Step 11 — Build the forest and let it vote

Each tree in the forest is grown exactly like our single tree — same `build_tree`, same Gini scoring. Only its ingredients differ: a bootstrap sample instead of the full data, and a random 6-word menu instead of all 9 words.

To classify a ticket, every tree answers on its own, and the forest goes with the **majority vote**.

One note on the code: because every tree's menu is drawn at random, the exact trees you get (and the exact vote counts) depend on the random draws. The *decisions* on clear tickets should still come out the same, because most reasonable trees agree on them.

In [11]:
def build_forest(tickets, number_of_trees, rng):
    forest = []
    for _ in range(number_of_trees):
        sample = bootstrap_sample(tickets, rng)    # each tree gets its own resampled data
        menu = rng.sample(CANDIDATE_WORDS, 6)      # ...and its own smaller menu of questions
        tree_for_sample = build_tree(sample, menu, 0, 3)
        forest.append(tree_for_sample)             # add the finished tree to the forest
    return forest


def forest_predict(forest, ticket_text):
    # collect one vote per tree
    votes = {}
    for tree_in_forest in forest:
        vote = predict(tree_in_forest, ticket_text)   # ask this tree for its answer
        if vote not in votes:
            votes[vote] = 0                           # first vote for this label
        votes[vote] += 1                              # count the vote
    # the label with the most votes wins
    winner = None
    winner_votes = 0
    for label in votes:
        if votes[label] > winner_votes:
            winner = label
            winner_votes = votes[label]
    return winner, votes


# grow a forest of 5 trees
forest = build_forest(TRAINING_TICKETS, 5, rng)

# look at two of the trees to see that they differ
print("Two of the five trees (notice they can ask different questions):")
print(format_tree(forest[0]))
print()
print(format_tree(forest[1]))
print()

# let the whole forest vote on two new tickets
test_tickets = [
    "My package never arrived and tracking is stuck",
    "I need a refund my card was charged twice",
]
for text in test_tickets:
    winner, votes = forest_predict(forest, text)
    print("Ticket:", text)
    print("  votes:", votes, "-> forest predicts:", winner)
    print()

Two of the five trees (notice they can ask different questions):
contains "login"?
  yes: predict account
  no:  contains "password"?
    yes: predict account
    no:  contains "order"?
      yes: predict shipping
      no:  predict shipping

contains "refund"?
  yes: predict billing
  no:  contains "password"?
    yes: predict account
    no:  predict shipping

Ticket: My package never arrived and tracking is stuck
  votes: {'shipping': 5} -> forest predicts: shipping

Ticket: I need a refund my card was charged twice
  votes: {'shipping': 1, 'billing': 4} -> forest predicts: billing



## Wrap-up

- A **decision tree** classifies by asking yes/no questions one at a time, and it chooses those questions from data, greedily, using Gini impurity.
- **Gini impurity** measures how mixed a pile of labels is. A question's score is the weighted Gini of the two piles it creates — **lower is better**.
- A tree grown without limits will carve special branches for individual tickets. That is **overfitting**, and **depth is the knob** that controls it.
- A **random forest** trains many trees — each on a bootstrap sample, each with a restricted menu of questions — and lets them **vote**. The trees' individual mistakes point in different directions, so the vote cancels much of the noise.
- Both models still only know the exact words on their menu: "delivery fee charged to the wrong card" fooled the tree, and a ticket containing none of the menu words falls through to the default leaf. Better features (Chapter 6) attack exactly this weakness.

**Next lesson:** gradient boosting — instead of many independent trees voting, trees are built one after another, each one correcting the mistakes of the trees before it.